# AquaCode: Water Safety Prediction with an SVM

**Step 1: Training (once)**: learn from many labelled water samples.
**Step 2: Prediction (every time after)**: give the model a new reading and get SAFE / UNSAFE.

The notebook runs as-is on a *stand-in dataset* that imitates your 35-sample experiment
(7 TDS levels x 5 pH levels, 20 readings per beaker). When you have real ESP32 data, upload your
CSV in Cell 1 and everything else stays the same.

CSV columns required: `sample_id, ph, tds, temp, safe` (safe = 1, unsafe = 0).

## Cell 1: Load data (your CSV, or the stand-in dataset)

In [ ]:
import numpy as np, pandas as pd, joblib, os

# ---- Option A: upload your real CSV (Colab only) ----
USE_REAL_CSV = False          # set to True after you have logged real readings
if USE_REAL_CSV:
    from google.colab import files
    uploaded = files.upload()                      # choose your CSV file
    df = pd.read_csv(list(uploaded.keys())[0])
else:
    # ---- Option B: stand-in data imitating the 35-beaker experiment ----
    rng = np.random.default_rng(42)
    tds_levels = [10, 100, 250, 500, 750, 1000, 1500]     # ppm (distilled -> salty)
    ph_levels  = [5.0, 6.8, 7.5, 8.2, 9.5]                # acidic -> alkaline
    rows, sid = [], 0
    for t in tds_levels:
        for p in ph_levels:
            sid += 1
            temp = rng.uniform(24, 30)                    # room temperature of that beaker
            for _ in range(20):                           # 20 readings per beaker
                rows.append({
                    "sample_id": sid,
                    "ph":  p + rng.normal(0, 0.10),               # sensor noise
                    "tds": t * (1 + rng.normal(0, 0.04)) + rng.normal(0, 8),
                    "temp": temp + rng.normal(0, 0.2),
                    # label comes from the RECIPE (what you mixed), not the noisy reading
                    "safe": int(6.5 <= p <= 8.5 and t <= 500),
                })
    df = pd.DataFrame(rows)

print(df.shape, "rows |", df.sample_id.nunique(), "samples")
print(df.groupby("safe").sample_id.nunique().rename({0:"unsafe samples", 1:"safe samples"}))
df.head()

## Step 1: TRAINING

Two things matter here:
* **Split by sample, not by row.** The 20 readings from one beaker are near-copies, so a normal random split would leak them into both train and test and give fake-perfect scores.
* **Scale the features.** TDS goes up to 1500 while pH spans about 4-10, so without scaling the SVM would only "see" TDS.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

FEATURES = ["ph", "tds", "temp"]
X, y, groups = df[FEATURES], df["safe"], df["sample_id"]

# hold out ~20% of the BEAKERS for an honest test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=3)
tr, te = next(gss.split(X, y, groups))
Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
print("train beakers:", groups.iloc[tr].nunique(), "| test beakers:", groups.iloc[te].nunique())

pipe = Pipeline([("scale", StandardScaler()),
                 ("svm", SVC(class_weight="balanced"))])

grid = GridSearchCV(
    pipe,
    {"svm__kernel": ["rbf", "poly"], "svm__C": [0.1, 1, 10, 100], "svm__gamma": ["scale", 0.1, 1]},
    cv=GroupKFold(n_splits=5),          # cross-validate by beaker too
    scoring="balanced_accuracy",
    n_jobs=-1,
)
grid.fit(Xtr, ytr, groups=groups.iloc[tr])
print("best settings:", grid.best_params_)

pred = grid.predict(Xte)
print("\nConfusion matrix (rows = truth, cols = predicted; order: unsafe, safe)")
print(confusion_matrix(yte, pred))
print(classification_report(yte, pred, target_names=["unsafe", "safe"]))

**How to read the report:** look at **recall for "unsafe"**. That is the share of truly unsafe samples the model caught. Calling unsafe water safe is the costly mistake.

Now retrain on *all* the data with the winning settings and save the model. This file is what Step 2 uses.

In [ ]:
final = grid.best_estimator_.fit(X, y)        # refit on every sample
joblib.dump(final, "svm_water.joblib")
print("saved: svm_water.joblib")

# Colab sessions are temporary. Copy the model to Google Drive so you keep it:
# from google.colab import drive; drive.mount('/content/drive')
# import shutil; shutil.copy("svm_water.joblib", "/content/drive/MyDrive/svm_water.joblib")

## Step 2: PREDICTION

Load the saved model and judge **new** water. The ESP32 gives many readings per second, so we **average a few readings** first (cheap sensors are noisy), then ask the model.
The rule-based WHO-style check is shown next to it, so students can compare the model with the rule.

In [ ]:
model = joblib.load("svm_water.joblib")

def rule_check(ph, tds):
    return int(6.5 <= ph <= 8.5 and tds <= 500)

def predict_water(readings):
    """readings: list of (ph, tds, temp) tuples, e.g. the last 10 seconds from the ESP32."""
    r = pd.DataFrame(readings, columns=FEATURES).mean()          # average out the noise
    x = pd.DataFrame([r.values], columns=FEATURES)
    label = "SAFE" if model.predict(x)[0] == 1 else "UNSAFE"
    margin = model.decision_function(x)[0]       # >0 leans SAFE, <0 leans UNSAFE; bigger = more sure
    rule = "SAFE" if rule_check(r.ph, r.tds) else "UNSAFE"
    print(f"pH {r.ph:.2f} | TDS {r.tds:.0f} ppm | {r.temp:.1f} C  ->  model: {label} "
          f"(score {margin:+.2f}) | WHO-style rule: {rule}")
    return label

# --- three new "unseen" waters (pretend these came from the ESP32) ---
tap_water   = [(7.4, 180, 27.0), (7.5, 185, 27.1), (7.4, 178, 27.0)]
salty_water = [(7.6, 1250, 27.2), (7.5, 1240, 27.1), (7.6, 1262, 27.2)]
acidic_wtr  = [(5.4, 220, 26.8), (5.5, 215, 26.9), (5.4, 224, 26.8)]

for name, data in [("tap", tap_water), ("salty", salty_water), ("acidic", acidic_wtr)]:
    print(name.ljust(7), end=": ")
    predict_water(data)

## Next steps
1. Replace the stand-in data with your real CSV (`USE_REAL_CSV = True`).
2. Compare with **Logistic Regression** and a **Decision Tree** on the same split (swap `SVC` in the pipeline).
3. Later: put `predict_water` behind a FastAPI `/api/predict` endpoint so the dashboard shows the model's answer live.